In [ ]:
import os
import pandas as pd
from netCDF4 import Dataset
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from scipy.interpolate import griddata
import matplotlib.ticker as mticker

In [ ]:
# To use PLUMBER2_GPP_common_utils, change directory to where it exists
os.chdir('/g/data/w97/mm3972/scripts/PLUMBER2/LSM_GPP_PLUMBER2')
from PLUMBER2_GPP_common_utils import *

### Check site names order

In [ ]:
def columns_name(model_in, var_name):
    
    df_obs  = pd.read_csv(f'./txt/NEE_annual_mean/NEE_annual_mean_obs.csv')
    site_names = df_obs['site_name']
    
    df  = pd.read_csv(f'./txt/{var_name}_annual_mean/{var_name}_annual_mean_{model_in}.csv')
    for i in np.arange(170):
        if df.loc[i, 'site_name'] != site_names.loc[i]:
            print('annual', df.loc[i, 'site_name'], site_names.loc[i])
            
    for month in np.arange(1,13):
        df  = pd.read_csv(f'./txt/{var_name}_monthly_mean/{var_name}_month{month}_mean_{model_in}.csv')
        for i in np.arange(170):
            if df.loc[i, 'site_name'] != site_names.loc[i]:
                print('annual', df.loc[i, 'site_name'], site_names.loc[i])
    return

In [ ]:
PLUMBER2_path_site = "/g/data/w97/mm3972/scripts/PLUMBER2/LSM_GPP_PLUMBER2/nc_files/AR-SLu.nc"
f                  = nc.Dataset(PLUMBER2_path_site, mode='r')

var_names          = ['NEE','GPP']

diff               = True
for var_name in var_names:
    model_list     = f.variables[f'{var_name}_models'][:]

    model_list     = model_list.tolist()
    model_list.append('obs')
    
    for model_in in model_list:
        columns_name(model_in, var_name)

### Put together

In [ ]:
def read_columns(var_name, model_in, month=None):
    
    if month == None:
        file_path = f'./txt/{var_name}_annual_mean/{var_name}_annual_mean_{model_in}.csv'
    else:
        file_path = f'./txt/{var_name}_monthly_mean/{var_name}_month{month}_mean_{model_in}.csv'
        
    if os.path.exists(file_path):
        df  = pd.read_csv(file_path)
        df = df[df['lat']!= 0]
        df_sorted = df.sort_values(by='site_name', ascending=True)
        return df_sorted[var_name], df_sorted['site_name']
    else:
        return np.nan, np.nan

In [ ]:
NEE_only_models      = ['1lin', '3km27', '6km729', '6km729lag', 'RF_eb', 'RF_raw', 'LSTM_eb', 
                       'LSTM_raw', 'NoahMPv401']

Both_NEE_GPP_models  = ['CABLE', 'CABLE-POP-CN', 'CHTESSEL_Ref_exp1', 'GFDL', 'MuSICA', 
                       'ORC2_r6593', 'ORC3_r8120', 'QUINCY', 'STEMMUS-SCOPE']

GPP_only_models      = ['CLM5a','JULES_GL9', 'JULES_GL9_withLAI']

model_full_list      = ['obs', '1lin', '3km27', '6km729', '6km729lag', 'RF_eb', 'RF_raw', 'LSTM_eb', 
                        'LSTM_raw', 'NoahMPv401', 'CABLE', 'CABLE-POP-CN', 'CHTESSEL_Ref_exp1', 
                        'GFDL', 'MuSICA', 'ORC2_r6593', 'ORC3_r8120', 'QUINCY', 'STEMMUS-SCOPE',
                        'CLM5a','JULES_GL9', 'JULES_GL9_withLAI']

filled_by_obs        = []
df_out               = None

#### Read all data ####
var_names = ['NEE','GPP']
months    = [None, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

for var_name in var_names:
    for model_in in model_full_list:
        for month in months:
            
            tmp, site_names = read_columns(var_name, model_in, month=month)
            
            if ~np.all(np.isnan(tmp)):
                if month == None:
                    varout_name = f'annual_{var_name}_{model_in}'
                else:
                    varout_name = f'month{month}_{var_name}_{model_in}'
                if df_out is None:
                    df_out = pd.DataFrame(site_names.values,columns=['site_name'])
                    df_out[varout_name] = tmp.values
                else:
                    df_out[varout_name] = tmp.values
                    
                
#### Calculate RS and fill GPP and NEE ####
for model_in in model_full_list:
    for month in months:
        if month == None:
            head = 'annual'
        else:
            head = f'month{month}'
            
        if f'{head}_GPP_{model_in}' in df_out.columns and f'{head}_NEE_{model_in}' in df_out.columns:
            df_out[f'{head}_RS_{model_in}'] = df_out[f'{head}_NEE_{model_in}'].values + df_out[f'{head}_GPP_{model_in}'].values 
        elif f'{head}_GPP_{model_in}' in df_out.columns:
            df_out[f'{head}_RS_{model_in}_derived']  = df_out[f'{head}_RS_obs']
            df_out[f'{head}_NEE_{model_in}_derived'] = df_out[f'{head}_RS_obs']-df_out[f'{head}_GPP_{model_in}']
        elif f'{head}_NEE_{model_in}' in df_out.columns:
            df_out[f'{head}_RS_{model_in}_derived']  = df_out[f'{head}_RS_obs']
            df_out[f'{head}_GPP_{model_in}_derived'] = df_out[f'{head}_RS_obs']-df_out[f'{head}_NEE_{model_in}']

df_out.to_csv(f'./txt/annual_monthly_mean_all.csv', index=False)
